# 6.4.5 Multimodal Embeddings and Retrieval

Cross-modal retrieval using numpy cosine similarity. Compute Recall@k and Mean Reciprocal Rank.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

Path('output').mkdir(exist_ok=True)

def unit_embed(seed, dim=64):
    v = np.random.default_rng(seed).standard_normal(dim)
    return v / (np.linalg.norm(v) + 1e-10)

def recall_at_k(sim, k):
    return sum(1 for i in range(len(sim)) if i in np.argsort(sim[i])[::-1][:k]) / len(sim)

n = 20; dim = 128
imgs = np.array([unit_embed(i*2, dim) for i in range(n)])
txts = np.array([unit_embed(i*2, dim) for i in range(n)])  # matched
txts_noisy = np.array([unit_embed(i*2 if i%3 else 999, dim) for i in range(n)])

sim_p = imgs @ txts.T; sim_n = imgs @ txts_noisy.T
for k in [1, 3, 5, 10]:
    print(f'Recall@{k}: perfect={recall_at_k(sim_p, k):.2f}, noisy={recall_at_k(sim_n, k):.2f}')

In [ ]:
ks = [1, 2, 3, 5, 10]
rp = [recall_at_k(sim_p, k) for k in ks]
rn = [recall_at_k(sim_n, k) for k in ks]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ks, rp, 'o-', color='steelblue', lw=2, label='Perfect')
axes[0].plot(ks, rn, 's--', color='tomato', lw=2, label='Noisy')
axes[0].set(xlabel='k', ylabel='Recall@k', title='Cross-Modal Retrieval'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

im = axes[1].imshow(sim_n[:10, :10], cmap='RdYlGn', vmin=-0.5, vmax=1)
axes[1].set(title='Cosine Sim Matrix (10x10)', xlabel='Text', ylabel='Image')
plt.colorbar(im, ax=axes[1])
plt.tight_layout()
plt.savefig('output/multimodal_retrieval.png')
print('Saved multimodal_retrieval.png')